# Tables 5 & 6: Protocol comparison at budget 10

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import os
import pandas as pd

from src.data_loader import (
    load_accessible_professions, load_latent_professions, load_multiturn_professions,
    load_multi_output_professions, try_load, warn_incomplete_coverage,
)
from src.metrics.ratios import unique_valid_at_budget
from src.metrics.bootstrap_ci import accessible_coverage_bootstrap

accessible = try_load(load_accessible_professions, label="accessible")
latent = try_load(load_latent_professions, expanded=True, label="latent (expanded)")
multiturn = try_load(load_multiturn_professions, label="multiturn")
multi_output = try_load(load_multi_output_professions, label="multi_output")

for name, df in [("accessible", accessible), ("latent", latent), ("multiturn", multiturn), ("multi_output", multi_output)]:
    if df is not None:
        warn_incomplete_coverage(df, label=name)


## Table 5: Unique valid names per model after budget=10, all 4 protocols

In [ ]:
BUDGET = 10
protocol_dfs = {"accessible": accessible, "latent": latent, "multiturn": multiturn, "multi_output": multi_output}
series = {}
for name, df in protocol_dfs.items():
    if df is None:
        print(f"Table 5: skipping column {name!r}: no data loaded.")
        continue
    series[name] = unique_valid_at_budget(df, BUDGET, group_cols=["model_version"]).groupby("model_version")["unique_valid"].sum()

table5 = pd.DataFrame(series) if series else pd.DataFrame()
table5


In [ ]:
if table5.empty:
    print("Skipping export: Table 5 has no data.")
else:
    out_path = "../data/results/metrics/table5_protocol_comparison_10.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    table5.to_csv(out_path)
    print("wrote", out_path)
